# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Youssof-Essam/Flyrank-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method: HistGradientBoostingClassifier** (scikit-learn)

**Why this fits Lane 2 (Refresh/Content Opportunity Scoring):**
- **Ranking task** -> precision@K is the metric; GBMs optimize ranking losses well
- **Tabular data** with mixed types (numeric + categorical) -> HGB handles natively
- **Speed** -> 10x faster than Random Forest on 500K+ rows, critical for Colab
- **NaN handling** -> native missing value support (no imputation needed for word_count)
- **Interpretability** -> built-in feature importance + SHAP compatible

**Baseline to beat**: precision@50 = 0.700 (stale_visible_page rule, client-holdout)

In [ ]:
# --- Token loader (Colab Secrets or .env) ---
import os
import sys

def get_hf_token():
    # 1. Try Colab secrets first
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    
    # 2. Fall back to .env file (local dev) - search from cwd up to repo root
    from pathlib import Path
    cwd = Path.cwd()
    env_path = cwd / ".env"
    while not env_path.exists() and cwd != cwd.parent:
        cwd = cwd.parent
        env_path = cwd / ".env"
    if env_path.exists():
        from dotenv import load_dotenv
        load_dotenv(env_path)
        token = os.getenv("HF_TOKEN")
        if token:
            return token
    
    # 3. Fall back to env var (already set in shell)
    token = os.getenv("HF_TOKEN")
    if token:
        return token
    
    raise RuntimeError("HF_TOKEN not found. Set in Colab Secrets or .env file.")

HF_TOKEN = get_hf_token()
print("HF_TOKEN loaded")

In [ ]:
# --- Load dimension tables via datasets library (handles gated access + splits) ---
from datasets import load_dataset
import pandas as pd
import os

print("Loading dimension tables via datasets library...")
dim_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train",
    token=HF_TOKEN
).to_pandas()

dim_clients = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train",
    token=HF_TOKEN
).to_pandas()

print(f"dim_content: {len(dim_content)} rows")
print(f"dim_clients: {len(dim_clients)} rows")

# Cache dimension tables locally
os.makedirs("work/outputs", exist_ok=True)
dim_content.to_parquet("work/outputs/dim_content.parquet", index=False)
dim_clients.to_parquet("work/outputs/dim_clients.parquet", index=False)
print("Cached dimension tables to work/outputs/")

# --- DuckDB for fact table (fast partition pruning) ---
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

# Cache directory
os.makedirs("work/outputs", exist_ok=True)

# --- Cache March 2026 fact partition (target/outcome month) ---
cache_path = "work/outputs/month_2026_03.parquet"
if not os.path.exists(cache_path):
    print("Loading and caching March 2026 partition (partition pruning)...")
    query = f"SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
    df_march = con.execute(query).df()
    df_march.to_parquet(cache_path, index=False)
    print(f"Cached {len(df_march)} rows to {cache_path}")
else:
    df_march = pd.read_parquet(cache_path)
    print(f"Loaded cached March 2026: {len(df_march)} rows")

# Load March 2026 for label construction
df_march = pd.read_parquet("work/outputs/month_2026_03.parquet")
print(f"March 2026 rows: {len(df_march)}")

In [ ]:
# --- Load feature window partitions (Dec 2025, Jan 2026, Feb 2026) ---
feature_months = ["2025-12", "2026-01", "2026-02"]
REL = "hf://datasets/FlyRank/internship-warehouse"

for m in feature_months:
    cache_path = f"work/outputs/month_{m}.parquet"
    if not os.path.exists(cache_path):
        print(f"Loading and caching {m} partition (partition pruning)...")
        con.execute(f"COPY (SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={m}/*.parquet')) TO '{cache_path}' (FORMAT PARQUET)")
        print(f"  Cached {m}")
    else:
        print(f"  Loaded cached {m}")

In [ ]:
# --- Build future-window label: 30-day decline in March 2026 ---
# Label: impressions in March 2026 / impressions in Dec-Feb < 0.8 (20% drop)

# 1. Aggregate March 2026 impressions per content
march_impressions = con.execute("""
SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) as impressions_mar26
FROM read_parquet('work/outputs/month_2026_03.parquet')
WHERE gsc_data_available = true
GROUP BY 1,2
""").df()

# 2. Aggregate feature window (Dec-Feb) impressions per content
feat_window = con.execute("""
SELECT 
    f.content_hash_id,
    f.client_hash_id,
    SUM(f.gsc_impressions) as impressions_90d,
    SUM(f.gsc_clicks) as clicks_90d,
    SUM(f.ga4_sessions) as sessions_90d,
    AVG(NULLIF(f.gsc_avg_position, 0)) as avg_position_90d,
    AVG(f.ga4_total_engagement_sec) as engagement_rate_90d
FROM (
    SELECT * FROM read_parquet('work/outputs/month_2025-12.parquet')
    UNION ALL
    SELECT * FROM read_parquet('work/outputs/month_2026-01.parquet')
    UNION ALL
    SELECT * FROM read_parquet('work/outputs/month_2026-02.parquet')
) f
WHERE f.gsc_data_available = true
GROUP BY 1,2
""").df()

# 3. Merge and compute label
labels = feat_window.merge(march_impressions, on=["content_hash_id", "client_hash_id"], how="left")
labels["impressions_mar26"] = labels["impressions_mar26"].fillna(0)
labels["is_declining_future"] = (labels["impressions_mar26"] / labels["impressions_90d"] < 0.8).astype(int)

print(f"Total content items: {len(labels)}")
print(f"Positive (declining future): {labels['is_declining_future'].sum()} ({labels['is_declining_future'].mean():.1%})")
print(f"Base rate: {labels['is_declining_future'].mean():.3f}")

In [ ]:
# --- Build feature frame with dim_content metadata ---
feat_query = """
SELECT 
    fw.content_hash_id,
    fw.client_hash_id,
    fw.impressions_90d,
    fw.clicks_90d,
    fw.sessions_90d,
    fw.avg_position_90d,
    fw.engagement_rate_90d,
    d.content_updated_date,
    d.word_count,
    d.content_created_date,
    d.content_type,
    d.main_intent
FROM (
    SELECT 
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_impressions) as impressions_90d,
        SUM(f.gsc_clicks) as clicks_90d,
        SUM(f.ga4_sessions) as sessions_90d,
        AVG(NULLIF(f.gsc_avg_position, 0)) as avg_position_90d,
        AVG(f.ga4_total_engagement_sec) as engagement_rate_90d
    FROM (
        SELECT * FROM read_parquet('work/outputs/month_2025-12.parquet')
        UNION ALL
        SELECT * FROM read_parquet('work/outputs/month_2026-01.parquet')
        UNION ALL
        SELECT * FROM read_parquet('work/outputs/month_2026-02.parquet')
) f
JOIN (
    SELECT DISTINCT content_hash_id, client_hash_id
    FROM read_parquet('work/outputs/month_2026_03.parquet')
) m ON f.content_hash_id = m.content_hash_id AND f.client_hash_id = m.client_hash_id
WHERE f.gsc_data_available = true
GROUP BY 1,2
) fw
JOIN read_parquet('work/outputs/dim_content.parquet') d
  ON fw.content_hash_id = d.content_hash_id
"""
feat_df = con.execute(feat_query).df()

# Merge labels
feat_df = feat_df.merge(labels[["content_hash_id", "client_hash_id", "is_declining_future", "impressions_mar26"]], 
                        on=["content_hash_id", "client_hash_id"], how="left")
feat_df["is_declining_future"] = feat_df["is_declining_future"].fillna(0).astype(int)

# Derived features
import numpy as np
feat_df["log_impressions_90d"] = np.log1p(feat_df["impressions_90d"])
feat_df["ctr_90d"] = np.where(feat_df["impressions_90d"] > 0, feat_df["clicks_90d"] / feat_df["impressions_90d"] * 100, 0)
feat_df["has_word_count"] = feat_df["word_count"].notna().astype(int)
feat_df["content_age_days"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat_df["content_created_date"])).dt.days
feat_df["days_since_last_update"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat_df["content_updated_date"])).dt.days

# Categorical encoding
feat_df["content_type"] = feat_df["content_type"].fillna("unknown").astype("category")
feat_df["main_intent"] = feat_df["main_intent"].fillna("unknown").astype("category")

print(f"Feature frame: {len(feat_df)} rows")
print(f"Positive rate: {feat_df['is_declining_future'].mean():.3f}")

In [ ]:
# --- Client-grouped + time-aware split ---
# Train: clients with data in Dec 2025 - Jan 2026
# Test: March 2026 (holdout) - already separated by label construction

# Get clients with sufficient history (train clients = those in Dec/Jan/Feb)
train_clients = feat_df["client_hash_id"].unique()

# Use client-grouped split: 80/20 on clients
np.random.seed(42)
test_clients = set(np.random.choice(train_clients, size=max(1, len(train_clients)//5), replace=False))
test_mask = feat_df["client_hash_id"].isin(test_clients)

# Feature columns
feature_cols = [
    "log_impressions_90d", "avg_position_90d", "ctr_90d",
    "days_since_last_update", "content_age_days",
    "word_count", "has_word_count",
    "engagement_rate_90d", "sessions_90d",
    "content_type", "main_intent"
]

X = feat_df[feature_cols].copy()
y = feat_df["is_declining_future"].copy()

X_train, X_test = X[~test_mask], X[test_mask]
y_train, y_test = y[~test_mask], y[test_mask]

print(f"Train: {len(X_train)} rows, {y_train.mean():.3f} positive")
print(f"Test: {len(X_test)} rows, {y_test.mean():.3f} positive")
print(f"Test clients: {len(test_clients)}")

In [ ]:
# --- Train HistGradientBoostingClassifier ---
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import precision_score, roc_auc_score, average_precision_score

clf = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.1,
    max_depth=6,
    min_samples_leaf=20,
    random_state=42,
    categorical_features=["content_type", "main_intent"],
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

print("Training model...")
clf.fit(X_train, y_train)
print("Training complete.")

# Predictions
y_pred_proba = clf.predict_proba(X_test)[:, 1]
y_pred = clf.predict(X_test)

In [ ]:
# --- Evaluation ---
from sklearn.metrics import precision_score, roc_auc_score, average_precision_score, recall_score

# Precision@50
top50_idx = pd.Series(y_pred_proba, index=y_test.index).nlargest(50).index
prec50 = y_test.loc[top50_idx].mean()

# Other metrics
roc_auc = roc_auc_score(y_test, y_pred_proba)
avg_prec = average_precision_score(y_test, y_pred_proba)
recall50 = y_test.loc[top50_idx].sum() / y_test.sum()
base_rate = y_test.mean()

print(f"Precision@50: {prec50:.3f} (baseline: 0.700)")
print(f"ROC-AUC: {roc_auc:.3f}")
print(f"Average Precision: {avg_prec:.3f}")
print(f"Recall@50: {recall50:.3f}")
print(f"Base rate: {base_rate:.3f}")
print(f"Model beats baseline: {prec50 > 0.700}")

In [ ]:
# --- Feature importance (permutation importance for HGB) ---
from sklearn.inspection import permutation_importance

result = permutation_importance(clf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": result.importances_mean
}).sort_values("importance", ascending=False)

print(importance.to_string(index=False))

# Save feature importance chart data
importance.to_csv("work/outputs/feature_importance.csv", index=False)

In [ ]:
# --- Calibration plot data ---
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10, strategy="uniform")
cal_df = pd.DataFrame({"prob_pred": prob_pred, "prob_true": prob_true})
cal_df.to_csv("work/outputs/calibration_curve.csv", index=False)
print("Calibration curve saved.")

In [ ]:
# --- Precision@K curve ---
def precision_at_k(y_true, y_scores, k):
    order = np.argsort(-y_scores)
    top_k = y_true.iloc[order[:k]]
    return top_k.mean() if len(top_k) > 0 else 0

ks = [10, 20, 30, 40, 50, 100, 200]
prec_curve = []
for k in ks:
    if k <= len(y_test):
        prec_curve.append({"k": k, "precision": precision_at_k(y_test, y_pred_proba, k)})

prec_df = pd.DataFrame(prec_curve)
prec_df.to_csv("work/outputs/precision_at_k_curve.csv", index=False)
print(prec_df.to_string(index=False))

In [ ]:
# --- Save model and metrics ---
import joblib
import json

os.makedirs("work/outputs", exist_ok=True)
joblib.dump(clf, "work/outputs/model.pkl")

metrics = {
    "precision_at_50": round(float(prec50), 3),
    "roc_auc": round(float(roc_auc), 3),
    "average_precision": round(float(avg_prec), 3),
    "recall_at_50": round(float(recall50), 3),
    "base_rate": round(float(base_rate), 3),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "test_clients": int(len(test_clients)),
    "positive_rate_train": float(y_train.mean()),
    "positive_rate_test": float(y_test.mean()),
    "beats_baseline": bool(prec50 > 0.700)
}

with open("work/outputs/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Model and metrics saved.")
print(json.dumps(metrics, indent=2))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.